# 11 (heavy) - Human-mouse gene maps for SAMap (MMseqs2)

Copies of the Project 23 WP2 scripts (https://github.com/fbastian/BH2026-23-single-cell, commit in UPSTREAM_COMMIT.txt;
our changes in 02_vs_upstream.diff / 03_vs_upstream.diff): GENCODE v48 (HLiCA) and vM18 (= Ensembl 93, Guilliams mm10-3.0.0)
transcripts, one transcript per gene, SAMap names (Symbol or Symbol__EnsemblID), MMseqs2 18.8cc5c translated search both
directions (-s 7.5 -e 1e-6 --min-length 30 ...). Record: ~45 min per direction on 20 threads.

**Disk: the search needs ~170 GB of temporary space under ROOT/tmp (on 11 Nov the vault dropped to 15 GB free).** The guard
below refuses to start with less than 200 GB free.

**Heavy notebook.** Nothing runs until you set `RUN_HEAVY = True` in the cell below. Long steps start in a detached
`screen` session with an exit sentinel (`WORK/logs/<name>.exit`); the wait cell polls the log. Re-run the light notebook named
below afterwards so that it uses these rebuilt outputs instead of the record's. Afterwards: notebooks 07 and 09.

**Runtime:** ~1.5 h + downloads, 20 threads, >= 200 GB temporary disk
**Envs used (each step runs as a subprocess in its own env):** rapids-sc (+ mmseqs2 binary from env mmseqs2)
**Kernel:** `masld_atlas` (orchestration only).

Outputs are written under `/mnt/masld_vault/cross_species_repro/` (mirror of `/mnt/masld_vault/cross_species/`, the outputs
of record). Inputs this notebook does not rebuild are linked from the record; see `repro_setup.py` for the safety rules.

In [ ]:
%run -i /home/ubuntu/liver_cross_species/notebooks/repro_setup.py
print("RECORD", RECORD); print("REPRO ", REPRO); print("WORK  ", WORK)

In [ ]:
RUN_HEAVY = False   # set True to run

In [ ]:
if RUN_HEAVY:   # nothing is touched unless the notebook is really run (review fix)
    guard(200)
    link("guilliams2022/01_raw")
    fresh("samap_maps/liver_hsmm/refs/raw", "samap_maps/liver_hsmm/refs/samap", "samap_maps/liver_hsmm/refs/mappings", "samap_maps/liver_hsmm/maps_mmseqs")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/01_download.sh
#!/bin/bash
# cDNA references matching our annotations: human GENCODE v48 (HLiCA, CxG schema 7.1.0), mouse GENCODE vM18 (= Ensembl 93 = 10x mm10-3.0.0)
R=/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm/refs/raw; cd $R
for u in https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_48/gencode.v48.transcripts.fa.gz \
         https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_mouse/release_M18/gencode.vM18.transcripts.fa.gz; do
  f=$(basename $u); [ -s $f ] || curl -s --retry 3 -o $f $u; gzip -t $f && echo "$f ok $(stat -c %s $f)" || echo "$f BAD"
done
sha256sum *.fa.gz > SHA256SUMS

In [ ]:
if RUN_HEAVY: run("01_download.sh", "rapids-sc")

In [ ]:
if RUN_HEAVY:   # 01_download.sh prints BAD but exits 0 on a broken file; check against the record checksums (review fix)
    rec_sums = dict(l.split()[::-1] for l in open(rec("samap_maps/liver_hsmm/refs/raw/SHA256SUMS")))
    new_sums = dict(l.split()[::-1] for l in open(rep("samap_maps/liver_hsmm/refs/raw/SHA256SUMS")))
    assert new_sums == rec_sums, f"downloads differ from the record: {new_sums} vs {rec_sums}"; print("downloads match the record checksums")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/02_build_samap_fastas_liver.py
from pathlib import Path
import gzip
import re
import textwrap

import pandas as pd
import scanpy as sc


# ============================================================
# CONFIGURATION
# ============================================================

# LIVER ADAPTATION (2026-11-11): copied from fbastian/BH2026-23-single-cell WP2/scripts (commit in
# UPSTREAM_COMMIT.txt). Changes: paths, species = human HLiCA (GENCODE v48) + mouse Guilliams 2022 (GENCODE vM18 =
# Ensembl 93), gene fields of our objects, GENCODE parser accepts human IDs. Naming + transcript choice unchanged.
ROOT = Path("/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm")

RAW = ROOT / "refs" / "raw"
OUT = ROOT / "refs" / "samap"
MAPS = ROOT / "refs" / "mappings"

OUT.mkdir(parents=True, exist_ok=True)
MAPS.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# MODIFY ONLY THESE PATH
# ------------------------------------------------------------

HUMAN_H5AD = Path(
    "/mnt/masld_vault/geo/cxg-059202e1-1f1b-483f-9151-f3a25a380c39/HLiCA_human_liver_all_cells.h5ad"
)

MOUSE_H5AD = Path(
    "/mnt/masld_vault/cross_species_repro/guilliams2022/01_raw/guilliams2022_fig1b_raw.h5ad"
)


CONFIG = {
    "hs": {
        "name": "human",
        "h5ad": HUMAN_H5AD,

        "fasta": (
            RAW
            / "gencode.v48.transcripts.fa.gz"
        ),

        "fasta_format": "gencode",

        "output": (
            OUT
            / "human.GRCh38.GENCODEv48.samap.fa"
        ),
    },

    "mm": {
        "name": "mouse",
        "h5ad": MOUSE_H5AD,

        "fasta": (
            RAW
            / "gencode.vM18.transcripts.fa.gz"
        ),

        "fasta_format": "gencode",

        "output": (
            OUT
            / "mouse.GRCm38.GENCODEvM18.samap.fa"
        ),
    },
}


# ============================================================
# UTILITIES
# ============================================================

def strip_version(identifier):
    """
    ENSDARG00000012345.7
          ->
    ENSDARG00000012345
    """

    identifier = str(identifier)

    return re.sub(
        r"\.\d+$",
        "",
        identifier,
    )


def fasta_reader(path):
    """
    Streaming FASTA reader for .fa.gz.
    """

    header = None
    sequence = []

    with gzip.open(path, "rt") as handle:

        for line in handle:

            line = line.rstrip()

            if line.startswith(">"):

                if header is not None:
                    yield header, "".join(sequence)

                header = line[1:]
                sequence = []

            else:
                sequence.append(line)

        if header is not None:
            yield header, "".join(sequence)


# ============================================================
# EXTRACT TRANSCRIPT / GENE IDS
# ============================================================

def parse_ensembl_header(header):
    """
    Expected Ensembl style:

    >ENSDART... chromosome:... gene:ENSDARG...
    """

    transcript_id = header.split()[0]

    match = re.search(
        r"(?:^|\s)gene:([^\s]+)",
        header,
    )

    if match is None:
        return None, None

    gene_id = strip_version(
        match.group(1)
    )

    transcript_id = strip_version(
        transcript_id
    )

    return transcript_id, gene_id


def parse_gencode_header(header):
    """
    GENCODE transcript FASTA style:

    >ENSMUST....|ENSMUSG....|...
    """

    first = header.split()[0]

    fields = first.split("|")

    if len(fields) < 2:
        return None, None

    transcript_id = strip_version(
        fields[0]
    )

    gene_id = strip_version(
        fields[1]
    )

    if not gene_id.startswith(
        ("ENSMUSG", "ENSG")
    ):
        return None, None

    return transcript_id, gene_id


# ============================================================
# CREATE DATASET GENE TABLE
# ============================================================

def load_gene_table(species, h5ad_file):

    print()
    print("=" * 70)
    print("Reading H5AD:", species)
    print("=" * 70)

    # backed mode:
    # we only need .var, not the huge expression matrix
    adata = sc.read_h5ad(
        h5ad_file,
        backed="r",
    )

    # liver objects: var_names = Ensembl IDs; symbols in var['feature_name'] (HLiCA, CxG) / var['symbol'] (mouse)
    if species == "hs":

        gene_id = pd.Series(
            adata.var_names.astype(str)
        )

        gene_symbol = pd.Series(
            adata.var["feature_name"].astype(str).values
        )

    elif species == "mm":

        gene_id = pd.Series(
            adata.var_names.astype(str)
        )

        gene_symbol = pd.Series(
            adata.var[
                "symbol"
            ].astype(str).values
        )

    else:
        raise ValueError(species)

    adata.file.close()

    gene_id = gene_id.map(
        strip_version
    )

    gene_symbol = (
        gene_symbol
        .astype(str)
        .str.strip()
    )


    # --------------------------------------------------------
    # Sanity checks
    # --------------------------------------------------------

    if gene_id.isna().any():
        raise RuntimeError(
            f"{species}: missing gene IDs"
        )

    if gene_symbol.isna().any():
        raise RuntimeError(
            f"{species}: missing gene symbols"
        )

    if (gene_symbol == "").any():
        raise RuntimeError(
            f"{species}: empty gene symbols"
        )


    # FASTA IDs are terminated by whitespace.
    # Gene symbols should therefore not contain spaces.
    has_space = gene_symbol.str.contains(
        r"\s",
        regex=True,
    )

    if has_space.any():

        print(
            gene_symbol[
                has_space
            ].head(20)
        )

        raise RuntimeError(
            f"{species}: gene symbols "
            "containing whitespace found."
        )


    # --------------------------------------------------------
    # Detect ambiguous symbols
    # --------------------------------------------------------

    duplicated_symbol = (
        gene_symbol
        .duplicated(
            keep=False
        )
    )


    samap_gene = gene_symbol.copy()


    # IMPORTANT:
    # every member of an ambiguous symbol group
    # gets its Ensembl Gene ID.
    samap_gene.loc[
        duplicated_symbol
    ] = (
        gene_symbol.loc[
            duplicated_symbol
        ]
        + "__"
        + gene_id.loc[
            duplicated_symbol
        ]
    )


    if not pd.Index(
        samap_gene
    ).is_unique:

        dup = samap_gene[
            samap_gene.duplicated(
                keep=False
            )
        ]

        raise RuntimeError(
            "SAMap names remain duplicated:\n"
            + str(dup.head(30))
        )


    table = pd.DataFrame(
        {
            "ensembl_gene_id":
                gene_id.values,

            "gene_symbol":
                gene_symbol.values,

            "samap_gene":
                samap_gene.values,

            "symbol_ambiguous":
                duplicated_symbol.values,
        }
    )


    table.to_csv(
        MAPS
        / f"{species}.gene_names.tsv",
        sep="\t",
        index=False,
    )


    print(
        "Genes:",
        len(table)
    )

    print(
        "Ambiguous genes:",
        table[
            "symbol_ambiguous"
        ].sum()
    )

    print(
        "Final SAMap names unique:",
        table[
            "samap_gene"
        ].is_unique
    )


    if table[
        "symbol_ambiguous"
    ].any():

        print()
        print(
            "Examples of ambiguous symbols:"
        )

        print(
            table.loc[
                table[
                    "symbol_ambiguous"
                ],
                [
                    "ensembl_gene_id",
                    "gene_symbol",
                    "samap_gene",
                ]
            ].head(20)
        )


    return table


# ============================================================
# BUILD ONE-SEQUENCE-PER-GENE FASTA
# ============================================================

def build_fasta(
    species,
    gene_table,
    input_fasta,
    output_fasta,
    fasta_format,
):

    print()
    print("=" * 70)
    print(
        "Building FASTA:",
        species
    )
    print("=" * 70)


    gene_to_name = dict(
        zip(
            gene_table[
                "ensembl_gene_id"
            ],
            gene_table[
                "samap_gene"
            ],
        )
    )


    if fasta_format == "ensembl":
        parser = parse_ensembl_header

    elif fasta_format == "gencode":
        parser = parse_gencode_header

    else:
        raise ValueError(
            fasta_format
        )


    # --------------------------------------------------------
    # best[gene_id] =
    # {
    #    transcript_id,
    #    sequence
    # }
    #
    # Criterion:
    # longest cDNA transcript
    # --------------------------------------------------------

    best = {}

    total_records = 0
    records_with_gene_id = 0
    records_matching_dataset = 0


    for header, sequence in fasta_reader(
        input_fasta
    ):

        total_records += 1

        transcript_id, gene_id = parser(
            header
        )

        if gene_id is None:
            continue

        records_with_gene_id += 1


        # Gene not used by the scRNA-seq dataset
        if gene_id not in gene_to_name:
            continue

        records_matching_dataset += 1


        current = best.get(
            gene_id
        )


        replace = False

        if current is None:
            replace = True

        elif len(sequence) > len(
            current["sequence"]
        ):
            replace = True

        elif (
            len(sequence)
            == len(
                current["sequence"]
            )
            and transcript_id
            < current["transcript_id"]
        ):
            # deterministic tie breaking
            replace = True


        if replace:

            best[gene_id] = {
                "transcript_id":
                    transcript_id,

                "sequence":
                    sequence,
            }


    print(
        "Total FASTA records:",
        total_records
    )

    print(
        "Records with parsed gene ID:",
        records_with_gene_id
    )

    print(
        "Records belonging to dataset genes:",
        records_matching_dataset
    )


    # --------------------------------------------------------
    # Write output in DATASET order
    # --------------------------------------------------------

    audit_rows = []

    n_written = 0


    with open(
        output_fasta,
        "w"
    ) as out:

        for row in gene_table.itertuples(
            index=False
        ):

            gene_id = (
                row.ensembl_gene_id
            )

            gene_symbol = (
                row.gene_symbol
            )

            samap_gene = (
                row.samap_gene
            )


            selected = best.get(
                gene_id
            )


            if selected is None:

                audit_rows.append(
                    {
                        "ensembl_gene_id":
                            gene_id,

                        "gene_symbol":
                            gene_symbol,

                        "samap_gene":
                            samap_gene,

                        "transcript_id":
                            None,

                        "length":
                            None,

                        "found":
                            False,
                    }
                )

                continue


            transcript_id = selected[
                "transcript_id"
            ]

            sequence = selected[
                "sequence"
            ]


            # ------------------------------------------------
            # IMPORTANT:
            #
            # first FASTA token is the exact SAMap gene name.
            #
            # Additional metadata comes AFTER whitespace,
            # so BLAST qseqid remains samap_gene.
            # ------------------------------------------------

            out.write(
                f">{samap_gene} "
                f"gene_id={gene_id} "
                f"transcript_id={transcript_id} "
                f"length={len(sequence)}\n"
            )


            for line in textwrap.wrap(
                sequence,
                width=80,
            ):
                out.write(
                    line + "\n"
                )


            n_written += 1


            audit_rows.append(
                {
                    "ensembl_gene_id":
                        gene_id,

                    "gene_symbol":
                        gene_symbol,

                    "samap_gene":
                        samap_gene,

                    "transcript_id":
                        transcript_id,

                    "length":
                        len(sequence),

                    "found":
                        True,
                }
            )


    audit = pd.DataFrame(
        audit_rows
    )


    audit_file = (
        MAPS
        / f"{species}.selected_transcripts.tsv"
    )

    audit.to_csv(
        audit_file,
        sep="\t",
        index=False,
    )


    missing = audit.loc[
        ~audit["found"]
    ]


    missing.to_csv(
        MAPS
        / f"{species}.missing_from_transcriptome.tsv",
        sep="\t",
        index=False,
    )


    print()
    print(
        "Genes written:",
        n_written
    )

    print(
        "Dataset genes:",
        len(gene_table)
    )

    print(
        "Genes missing from FASTA:",
        len(missing)
    )


    coverage = (
        100
        * n_written
        / len(gene_table)
    )


    print(
        f"Coverage: {coverage:.2f}%"
    )

    print()
    print(
        "FASTA:",
        output_fasta
    )

    print(
        "Audit table:",
        audit_file
    )


# ============================================================
# MAIN
# ============================================================

for species, cfg in CONFIG.items():

    gene_table = load_gene_table(
        species,
        cfg["h5ad"],
    )


    build_fasta(
        species=species,

        gene_table=gene_table,

        input_fasta=cfg["fasta"],

        output_fasta=cfg["output"],

        fasta_format=cfg[
            "fasta_format"
        ],
    )


print()
print("=" * 70)
print("DONE")
print("=" * 70)

In [ ]:
if RUN_HEAVY: run("02_build_samap_fastas_liver.py", "rapids-sc")

In [ ]:
%%writefile /home/ubuntu/liver_cross_species/notebooks/work/03_build_mmseqs_maps_liver.py
#!/usr/bin/env python3
"""Continue 02_build_samap_fastas.py: build mouse/zebrafish MMseqs2 maps.

Reuses the existing selected cDNA sequences and SAMap gene names unchanged.
Run in your SAMap environment after installing MMseqs2:
    conda install -c conda-forge -c bioconda mmseqs2
    python scripts/03_build_mmseqs_maps.py

Documentation: https://github.com/soedinglab/MMseqs2/wiki#translated-sequence-searching

Identifier fix: export full FASTA headers, then take their first token. MMseqs2
accession parsing can otherwise truncate custom names such as 'ucp3' to 'p3'.
On failure, working files are retained in ROOT/tmp instead of being deleted.
"""

from pathlib import Path
import csv
import math
import shutil
import subprocess
import tempfile


# Same project directory and FASTA files as your scripts/notebook.
# LIVER ADAPTATION 2026-11-11: copied from fbastian/BH2026-23-single-cell WP2/scripts (UPSTREAM_COMMIT.txt);
# changes: ROOT, species pair hs/mm and file names, THREADS. Search parameters unchanged.
ROOT = Path("/mnt/masld_vault/cross_species_repro/samap_maps/liver_hsmm")
FASTAS = {
    "hs": ROOT / "refs/samap/human.GRCh38.GENCODEv48.samap.fa",
    "mm": ROOT / "refs/samap/mouse.GRCm38.GENCODEvM18.samap.fa",
}
OUT = ROOT / "maps_mmseqs" / "hsmm"
THREADS = 20  # TotalVI shares the VM
EVALUE = 1e-6
SENSITIVITY = 7.5
# Prefilter candidates per translated query; increase for a sensitivity check.
MAX_SEQS = 10000
# 'query,target' invoke MMseqs2 accession parsing; full headers bypass that parsing.
FIELDS = "qheader,theader,pident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits"


def run(*arguments):
    subprocess.run([str(arg) for arg in arguments], check=True)


def fasta_names(path):
    """Read the first header token, exactly as used by your existing BLAST maps."""
    names = []
    with path.open() as handle:
        for line in handle:
            if line.startswith(">"):
                names.append(line[1:].split()[0])
    if not names or len(names) != len(set(names)):
        raise ValueError(f"Empty FASTA or duplicate gene names: {path}")
    return set(names)


def keep_best_alignment(raw_path, output_path, query_names, target_names):
    """Collapse ORF/frame hits to the best alignment per directed gene pair.

    Keep multiple target genes per query. SAMap handles reciprocity itself.
    Preserve the selected alignment's real E-value and bit score.
    """
    best = {}
    with raw_path.open() as handle:
        for row in csv.reader(handle, delimiter="\t"):
            if len(row) != 12:
                raise ValueError("Expected 12 alignment columns")
            # Script 02 writes: >samap_gene gene_id=... transcript_id=... length=...
            # Keep the exact first token, including colons and '__EnsemblGeneID'.
            if not row[0].split() or not row[1].split():
                raise ValueError("Empty query or target FASTA header in alignment output")
            query, target = row[0].split()[0], row[1].split()[0]
            if query not in query_names or target not in target_names:
                raise ValueError(f"Full FASTA headers do not match the input gene names: {query}, {target}")
            row[0], row[1] = query, target
            evalue, bits = float(row[10]), float(row[11])
            if not math.isfinite(evalue) or not math.isfinite(bits) or evalue < 0 or bits <= 0:
                raise ValueError("Invalid E-value or bit score")
            if evalue > EVALUE:
                continue
            pair = (query, target)
            # Highest bit score, then lowest E-value; deterministic exact ties.
            candidate = (bits, -evalue, tuple(row))
            if pair not in best or candidate > best[pair]:
                best[pair] = candidate
    if not best:
        raise ValueError(f"No significant gene pairs in {raw_path}")
    with output_path.open("w", newline="") as handle:
        writer = csv.writer(handle, delimiter="\t", lineterminator="\n")
        writer.writerows(best[pair][2] for pair in sorted(best))
    print(f"{output_path.name}: {len(best):,} gene pairs", flush=True)


def main():
    if shutil.which("mmseqs") is None:
        raise RuntimeError("MMseqs2 is missing; install it with the command in this script's header")
    if OUT.exists():
        raise FileExistsError(f"Output already exists: {OUT}; choose a different OUT for a new run")
    names = {sid: fasta_names(path) for sid, path in FASTAS.items()}
    temporary_root = ROOT / "tmp"
    temporary_root.mkdir(parents=True, exist_ok=True)

    work = Path(tempfile.mkdtemp(prefix="mmseqs_hs_mm_", dir=temporary_root))
    print(f"Working directory: {work}", flush=True)
    try:
        for sid, path in FASTAS.items():
            # Explicit nucleotide databases: do not interpret cDNA as protein.
            run("mmseqs", "createdb", path, work / sid, "--dbtype", 2)

        for query, target in [("hs", "mm"), ("mm", "hs")]:
            result = work / f"{query}_{target}_result"
            raw = work / f"{query}_{target}_raw.tsv"
            run("mmseqs", "search", work / query, work / target, result,
                work / f"tmp_{query}_{target}",
                "--search-type", 2, "-s", SENSITIVITY, "-e", EVALUE,
                "--min-length", 30, "--orf-start-mode", 1, "--translation-table", 1,
                "--max-seqs", MAX_SEQS, "--max-accept", 1000000,
                "--max-rejected", 1000000, "-a", "--threads", THREADS)
            run("mmseqs", "convertalis", work / query, work / target, result, raw,
                "--search-type", 2, "--format-output", FIELDS, "--threads", THREADS)
            keep_best_alignment(raw, work / f"{query}_to_{target}.txt",
                                names[query], names[target])

        # Publish the two finished maps together, separately from your BLAST maps.
        OUT.mkdir(parents=True)
        for filename in ("hs_to_mm.txt", "mm_to_hs.txt"):
            shutil.copyfile(work / filename, OUT / filename)
        with (OUT / "mmseqs_version.txt").open("w") as handle:
            subprocess.run(["mmseqs", "version"], check=True, stdout=handle)
    except BaseException:
        print(f"Run interrupted or failed. Working files retained in: {work}", flush=True)
        raise
    else:
        shutil.rmtree(work)
    print(f"Finished. SAMap f_maps: {OUT.parent}/")


if __name__ == "__main__":
    main()

In [ ]:
launch("mmseqs", [f"PATH={ENVS}/mmseqs2/bin:$PATH {PY['rapids-sc']} -I {WORK}/03_build_mmseqs_maps_liver.py > {LOGS}/03_mmseqs.log 2>&1"])
wait_for("mmseqs", log=f"{LOGS}/03_mmseqs.log", every=300)